# QLoRA fine-tune: Mistral-7B-Instruct on medical Q&A

Runs on a **free Colab T4 (16GB)**. Runtime → Change runtime type → **T4 GPU**.

Before running: accept the license at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 and have a HF **write** token ready.

⚠️ Educational only — not medical advice.

In [ ]:
!pip install -q torch==2.4.0 transformers==4.46.2 peft==0.13.2 trl==0.12.1 \
    accelerate==1.1.1 bitsandbytes==0.44.1 datasets==3.1.0 huggingface_hub==0.26.2 sentencepiece==0.2.0

In [ ]:
from huggingface_hub import login
login()  # paste your HF write token

## Config

In [ ]:
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
HUB_REPO   = "mohitxshukla/medqa-mistral-7b-qlora"   # <-- set to your HF username
TRAIN_SIZE = 8000
EVAL_SIZE  = 500
MAX_SEQ_LEN = 1024

SYSTEM_PROMPT = (
    "You are a careful medical information assistant. Give clear, evidence-based "
    "answers in plain language. You do not provide diagnoses or prescriptions, and "
    "you remind the user to consult a licensed clinician for personal medical decisions."
)

def build_training_text(q, a):
    return f"<s>[INST] {SYSTEM_PROMPT}\n\n{q.strip()} [/INST] {a.strip()}</s>"

## Data — load, format, dedup, split

In [ ]:
from datasets import load_dataset

raw = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

def to_text(r):
    q, a = (r.get("input") or "").strip(), (r.get("output") or "").strip()
    return {"text": build_training_text(q, a), "question": q, "answer": a}

ds = raw.map(to_text, remove_columns=raw.column_names)
ds = ds.filter(lambda r: len(r["answer"]) >= 20 and len(r["question"]) > 0)

seen, keep = set(), []
for i, q in enumerate(ds["question"]):
    if q.lower() not in seen:
        seen.add(q.lower()); keep.append(i)
ds = ds.select(keep).shuffle(seed=42).select(range(TRAIN_SIZE + EVAL_SIZE))
eval_ds  = ds.select(range(EVAL_SIZE))
train_ds = ds.select(range(EVAL_SIZE, len(ds)))
print(len(train_ds), len(eval_ds)); print(train_ds[0]["text"][:400])

## Load base model in 4-bit + tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
model.config.use_cache = False

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

## Attach LoRA + train

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

sft = SFTConfig(
    output_dir="out/medqa-mistral-7b-qlora", num_train_epochs=1,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_ratio=0.03, max_grad_norm=0.3, fp16=True, logging_steps=10,
    save_strategy="epoch", max_length=MAX_SEQ_LEN, dataset_text_field="text",
    packing=False, report_to="none",
)
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds,
                     peft_config=peft_config, processing_class=tok)
trainer.train()

## Save + push adapter to the Hub

In [ ]:
trainer.save_model("out/medqa-mistral-7b-qlora")
tok.save_pretrained("out/medqa-mistral-7b-qlora")
trainer.model.push_to_hub(HUB_REPO)
tok.push_to_hub(HUB_REPO)
print(f"Pushed: https://huggingface.co/{HUB_REPO}")

## Quick sanity generation

In [ ]:
q = "What are the early warning signs of type 2 diabetes?"
prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\n{q} [/INST]"
ids = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=256, do_sample=True, temperature=0.3, top_p=0.9)
print(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

## Next: evaluate (before/after)
Clone the repo in Colab and run `python src/evaluate.py --base $BASE_MODEL --adapter $HUB_REPO --n 200 --examples 5`, then paste the numbers from `eval/results_template.md` into `model_card/README.md`.